In [1]:
from pymmcore_plus import CMMCorePlus
from useq import MDAEvent

# Create the core instance.
mmc = CMMCorePlus.instance()
# Load the correct configuration file. NB! If a new configuration file is created, change the path to the new configuration file
mmc.loadSystemConfiguration(r"\Users\Hannah\AppData\Local\pymmcore-plus\pymmcore-plus\mm\Micro-Manager_2.0.3_20260429\configuration\MMConfig_test.cfg")  

# Create a super-simple sequence, with one event
mda_sequence = [MDAEvent()] 

# Run it!
mmc.run_mda(mda_sequence)

OSError: Line 68: Property,Core,Initialize,1
Error in device "Camera-1": [PVCAM] ERR: pl_cam_open failed, pvErr:195, pvMsg:'Driver device failed to open (PL_ERR_DDI_DEVICE_OPEN_FAILED)' (20195)



In [ ]:
#lets try to take a more advanced event, by changing the parameters of the camera
mda_sequence_2 =[MDAEvent(exposure=100, x_pos=10, y_pos =10)]

mmc.run_mda(mda_sequence_2)

In [ ]:
#Now let's try to take multiple images with the camera using multiple MDAEvents
while True:
    try:
        Acq_n = int(input("How many snaps should be taken? "))
        if isinstance(Acq_n, int):
            print('Number of snaps is an integer')
            break
    except ValueError:
        print("Not an integer. Please enter an integer value")

mda_sequence_3=[]
for i in range(Acq_n):
    xpos = i*2
    ypos = i*1
    mda_sequence_3 +=[MDAEvent(exposure=100, x_pos=xpos, y_pos =ypos)]
    
mmc.run_mda(mda_sequence_3)

In [ ]:
#The previous code is easy to read and works for making multiple events, but it can be helpful to use MDASequence instead
from useq import MDASequence

while True:
    try:
        Interval = int(input("How much time between each image? (ms) "))
        if isinstance(Acq_n, int):
            print('It is a valid time')
            break
    except ValueError:
        print("Not an integer. Please enter an integer value")
while True:
    try:
        Loops = int(input("How many images? "))
        if isinstance(Acq_n, int):
            print('It is a valid amount')
            break
    except ValueError:
        print("Not an integer. Please enter an integer value")


mda_sequence_4 = MDASequence(
    time_plan={"interval": Interval, "loops": Loops},
    channels=[
        {"config": "DAPI", "exposure": 50},
        {"config": "FITC", "exposure": 80},
    ]
)
mmc.run_mda(mda_sequence_4)

In [2]:
#One of the most important parts is the saving of the files, of course. We want to save it in tiff to start with
from pymmcore_plus import CMMCorePlus
from useq import MDAEvent
from useq import MDASequence
import useq
import numpy as np
import tifffile as tiff

@mmc.mda.events.frameReady.connect 
def on_frame(image: np.ndarray, event: useq.MDAEvent):
    # do what you want with the data
    print(
        f"received frame: {image.shape}, {image.dtype} "
        f"@ index {event.index}, z={event.z_pos}"
    )
    tiff.imwrite('temp.tif', image, photometric='minisblack')



mda_sequence_5 = MDASequence(
    time_plan={"interval": 1, "loops": 2},
    channels=[
        {"config": "MMConfig_test", "exposure": 50},
        {"config": "FITC", "exposure": 80},
    ]
)
mmc.run_mda(mda_sequence_5)



2026-05-19 15:56:37,527 - pymmcore-plus - INFO - (_runner.py:310) MDA Started: channels=(Channel(config='MMConfig_test', exposure=50.0), Channel(config='FITC', exposure=80.0)) time_plan=TIntervalLoops(interval=datetime.timedelta(seconds=1), loops=2)


<Thread(Thread-4 (run), started 29172)>

2026-05-19 15:56:37,532 - pymmcore-plus - INFO - (_runner.py:278) index={'t': 0, 'c': 0} channel=Channel(config='MMConfig_test') exposure=50.0 min_start_time=0.0 reset_event_timer=True
2026-05-19 15:56:37,533 - pymmcore-plus - WARNING - (_engine.py:255) Failed to set channel. Preset "MMConfig_test" of configuration group "Channel" does not exist
2026-05-19 15:56:37,650 - pymmcore-plus - INFO - (_runner.py:278) index={'t': 0, 'c': 1} channel=Channel(config='FITC') exposure=80.0 min_start_time=0.0
2026-05-19 15:56:37,652 - pymmcore-plus - WARNING - (_engine.py:255) Failed to set channel. Preset "FITC" of configuration group "Channel" does not exist


received frame: (3200, 3200), uint16 @ index {'t': 0, 'c': 0}, z=None
received frame: (3200, 3200), uint16 @ index {'t': 0, 'c': 1}, z=None


2026-05-19 15:56:38,532 - pymmcore-plus - INFO - (_runner.py:278) index={'t': 1, 'c': 0} channel=Channel(config='MMConfig_test') exposure=50.0 min_start_time=1.0
2026-05-19 15:56:38,533 - pymmcore-plus - WARNING - (_engine.py:255) Failed to set channel. Preset "MMConfig_test" of configuration group "Channel" does not exist
2026-05-19 15:56:38,657 - pymmcore-plus - INFO - (_runner.py:278) index={'t': 1, 'c': 1} channel=Channel(config='FITC') exposure=80.0 min_start_time=1.0
2026-05-19 15:56:38,658 - pymmcore-plus - WARNING - (_engine.py:255) Failed to set channel. Preset "FITC" of configuration group "Channel" does not exist
2026-05-19 15:56:38,807 - pymmcore-plus - INFO - (_runner.py:403) MDA Finished: channels=(Channel(config='MMConfig_test', exposure=50.0), Channel(config='FITC', exposure=80.0)) time_plan=TIntervalLoops(interval=datetime.timedelta(seconds=1), loops=2)


received frame: (3200, 3200), uint16 @ index {'t': 1, 'c': 0}, z=None
received frame: (3200, 3200), uint16 @ index {'t': 1, 'c': 1}, z=None


In [9]:
>>> data = tiff.imread('temp.tif')
>>> data.shape



(3200, 3200)